# Семинар 7. Обучение word2vec

**Курс:** «Интеллектуальный анализ текстов: от статистических моделей языка до языковых агентов». Семинар относится к Лекции 4 «Векторные представления слов (word embeddings)».

## Цель

Реализовать SGNS (Skip-gram with Negative Sampling) «с нуля» для понимания механики обучения, а также обучить word2vec-подобную модель на корпусе и исследовать полученное пространство эмбеддингов.

## Задачи семинара

1. Реализовать SGNS с негативной выборкой на небольшом корпусе (без библиотек, на NumPy/PyTorch).
2. Обучить word2vec-подобную модель (Skip-gram) на корпусе средствами PyTorch и сравнить с самописной NumPy-реализацией.
3. Проверить лингвистические аналогии на обученных эмбеддингах (метод 3CosAdd).
4. Визуализировать эмбеддинги через t-SNE/UMAP и проинтерпретировать кластеры.

## Ход занятия

1. Подготовка небольшого русскоязычного синтетического/открытого корпуса, построение словаря и таблицы частот для negative sampling.
2. Реализация форвард-прохода и градиентного шага SGNS вручную (NumPy).
3. Обучение Skip-gram-модели на PyTorch с сопоставимыми гиперпараметрами, сравнение с самописной реализацией.
4. Тест аналогий (например, «король − мужчина + женщина ≈ королева» и русские аналоги).
5. Визуализация слов корпуса через t-SNE/UMAP, обсуждение кластеров по темам/частям речи.

**Замечание об окружении.** В силлабусе семинар предполагает также обучение word2vec средствами библиотеки `gensim`. В данной программной среде используется **Python 3.14**, а `gensim` (через зависимость `scipy`/`Cython`-биндинги старых версий) **не устанавливается и не импортируется** на этой версии интерпретатора — при попытке `import gensim` возникает `ModuleNotFoundError`. Ноутбук обёрнут в `try/except`: если `gensim` доступен — используется его `Word2Vec`; если нет (наш случай) — печатается инструкция по установке, и в качестве «библиотечного» аналога Skip-gram используется **собственная реализация SGNS на PyTorch** (`nn.Embedding` + `nn.Embedding` + `BCEWithLogitsLoss`, что эквивалентно тому, что делает gensim внутри на C-уровне). Дополнительно рассмотрен классический **baseline матричной факторизации** — SVD над (сдвинутой) PPMI-матрицей совместных встречаемостей, который по теореме Levy & Goldberg (2014) эквивалентен неявной факторизации той же статистики, что и SGNS.


## Краткая теория

### 1. Гипотеза дистрибутивности и PMI

Гипотеза дистрибутивности: слова, встречающиеся в похожих контекстах, имеют похожее значение. Формализация совместной встречаемости слова $w$ и контекстного слова $c$ через **точечную взаимную информацию (PMI)**:
$$
\mathrm{PMI}(w,c) = \log \frac{P(w,c)}{P(w)P(c)} \approx \log\frac{\#(w,c)\cdot |D|}{\#(w)\cdot\#(c)},
$$
где $\#(w,c)$ — число совместных встреч в окне, $\#(w),\#(c)$ — частоты слов, $|D|$ — общее число пар в корпусе. Положительное PMI означает, что слова встречаются вместе чаще, чем предсказывает независимость.

### 2. Skip-gram с негативной выборкой (SGNS)

Для пары «слово-контекст» $(w,c)$ с векторами $v_w, v_c \in \mathbb{R}^d$ (два разных представления: «входное» и «выходное») целевая функция на одном обучающем примере:
$$
\mathcal{L}(w,c) = \log\sigma(v_w^\top v_c) + \sum_{i=1}^{k}\mathbb{E}_{c_i\sim P_n}\left[\log\sigma(-v_w^\top v_{c_i})\right],
$$
где $\sigma(x) = 1/(1+e^{-x})$, $P_n$ — распределение негативной выборки (обычно $\propto U(w)^{3/4}$, где $U$ — униграммная частота), $k$ — число негативных примеров на одну позитивную пару. Модель учится делать **скалярное произведение** векторов реально встретившейся пары большим, а для случайных «шумовых» пар — малым, избегая полного softmax по всему словарю (сложность $O(|V|)$ заменяется на $O(k)$). Обратите внимание: SGNS оптимизирует именно скалярное произведение, а не косинус — большое $v_w^\top v_c$ не обязательно означает высокую косинусную близость (она дополнительно зависит от норм векторов). Косинусная близость, которой мы пользуемся при анализе соседей и аналогий, — это принятая на практике *пост-хок* метрика, а не то, что модель максимизирует напрямую.

**Градиент по $v_w$** (для одной позитивной пары и $k$ негативных $c_1,\dots,c_k$):
$$
\frac{\partial \mathcal{L}}{\partial v_w} = \bigl(1-\sigma(v_w^\top v_c)\bigr)v_c - \sum_{i=1}^k \sigma(v_w^\top v_{c_i})\,v_{c_i}.
$$

### 3. Связь SGNS и матричной факторизации (теорема Levy & Goldberg, 2014)

При достаточной размерности эмбеддингов и оптимальном решении точка оптимума SGNS удовлетворяет
$$
v_w^\top v_c = \mathrm{PMI}(w,c) - \log k,
$$
то есть SGNS неявно факторизует матрицу **сдвинутой PPMI** (Shifted Positive PMI): $\mathrm{SPPMI}(w,c) = \max(0,\ \mathrm{PMI}(w,c)-\log k)$. Явный аналог — матричный метод: построить PPMI-матрицу и найти её низкоранговое приближение через **SVD**:
$$
M_{\mathrm{PPMI}} \approx U_d \Sigma_d V_d^\top, \qquad v_w := U_d[w,:] \cdot \sqrt{\Sigma_d}
$$
(берутся $d$ старших сингулярных векторов). Это и есть baseline «PPMI-SVD» в этом семинаре — детерминированная альтернатива стохастическому обучению SGNS, дающая качественно похожие эмбеддинги.

### 4. Аналогии: метод 3CosAdd

Аналогия «$a$ относится к $b$, как $c$ относится к $d$» (например, «мужчина:король = женщина:королева») проверяется через
$$
\hat d = \arg\max_{d' \in V\setminus\{a,b,c\}} \cos\left(v_b - v_a + v_c,\; v_{d'}\right), \qquad \cos(x,y) = \frac{x^\top y}{\|x\|\,\|y\|}.
$$
Интуиция: если семантическое отношение (например, «страна → столица») линейно кодируется в векторном пространстве, разность векторов $v_b - v_a$ должна быть примерно одинаковой для разных пар $(a,b)$ с этим отношением, поэтому прибавление этой разности к $v_c$ должно указывать на искомое слово $d$.

### 5. Визуализация: t-SNE и UMAP

Оба метода — нелинейные методы снижения размерности для визуализации высокоразмерных векторов на плоскости, сохраняющие **локальную** структуру окружения (близкие точки в исходном пространстве остаются близкими на карте), но **не** сохраняющие глобальные расстояния и масштаб — интерпретировать нужно только относительное расположение кластеров, а не абсолютные координаты или площади.


### Дополнение к теории: CBOW (Continuous Bag-of-Words)

Вторая архитектура word2vec (Mikolov et al., 2013) наряду со skip-gram. Если skip-gram предсказывает **контекст по слову** ($P(c \mid w)$ для каждого контекстного слова в окне), то CBOW — наоборот, **слово по контексту**: усреднённый вектор контекстных слов предсказывает центральное слово. Для окна $C = \{c_1,\dots,c_{2m}\}$ вокруг слова $w$:
$$
\bar v_C = \frac{1}{2m}\sum_{c \in C} v_c, \qquad
P(w \mid C) = \frac{\exp(\bar v_C^\top u_w)}{\sum_{w' \in V}\exp(\bar v_C^\top u_{w'})},
$$
и с негативной выборкой цель на одном примере полностью аналогична SGNS с заменой $v_w \to \bar v_C$:
$$
\mathcal{L}_{\text{CBOW}}(w, C) = \log\sigma(\bar v_C^\top u_w) + \sum_{i=1}^{k}\log\sigma(-\bar v_C^\top u_{w_i}), \qquad w_i \sim P_n.
$$

**Отличия от skip-gram.** CBOW делает один прогноз на окно (skip-gram — по прогнозу на каждую пару), поэтому обучается быстрее; усреднение контекста сглаживает сигнал, из-за чего CBOW лучше ведёт себя на частых словах, а skip-gram обычно даёт более качественные векторы **редких** слов и лучше работает на небольших корпусах (больше обучающих примеров с каждой позиции). Реализация CBOW предлагается в опциональной задаче 6* ниже.


In [ ]:
# Импорты и фиксация seed
import warnings
warnings.filterwarnings("ignore")

import re
import math
import random
from collections import Counter

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    import torch.nn as nn
    TORCH_OK = True
    torch.manual_seed(SEED)
except ImportError:
    TORCH_OK = False

print("PyTorch доступен:", TORCH_OK)
print("Seed зафиксирован:", SEED)


In [ ]:
# Попытка импортировать gensim (см. замечание об окружении в шапке ноутбука)
try:
    import gensim
    GENSIM_OK = True
    print("gensim найден, версия:", gensim.__version__)
except ImportError as e:
    GENSIM_OK = False
    print("gensim НЕ доступен в этой среде:", repr(e))
    print("Инструкция по установке (в среде с Python <=3.11): pip install gensim")
    print("В этом ноутбуке вместо gensim.Word2Vec используется собственная реализация")
    print("Skip-gram на PyTorch (раздел «Демо-задача 2»), эквивалентная по духу.")


In [ ]:
# Небольшой русскоязычный синтетический корпус (тематически расслоенный,
# чтобы после обучения были видны содержательные кластеры при визуализации).
# Данные генерируются кодом (не скачиваются из интернета) — компактный контролируемый
# сэмпл, характерный для учебных SGNS-семинаров с ограниченным CPU-бюджетом.

random.seed(SEED)

TEMPLATES = {
    "royalty": [
        "король правил страной мудро",
        "королева была женой короля",
        "принц наследник трона короля",
        "принцесса дочь короля и королевы",
        "царь и царица жили во дворце",
        "король и королева правили государством",
        "принц женился на принцессе",
        "царь издал указ о войне",
        "королева короновала нового короля",
        "дворец короля охраняли рыцари",
    ],
    "animals": [
        "кошка поймала мышь во дворе",
        "собака охраняла дом хозяина",
        "кошка и собака жили в одном доме",
        "мышь пряталась от кошки",
        "волк охотился на зайца в лесу",
        "заяц убежал от волка в лес",
        "лиса поймала зайца в поле",
        "собака гналась за лисой по лесу",
        "кошка спала на диване весь день",
        "волк и лиса охотились вместе",
    ],
    "capitals": [
        "москва столица россии",
        "париж столица франции",
        "берлин столица германии",
        "рим столица италии",
        "москва крупнейший город россии",
        "париж крупный город франции",
        "берлин крупный город германии",
        "рим древний город италии",
        "россия граничит с германией и францией",
        "франция и италия соседние страны европы",
    ],
    "food": [
        "хлеб пекут из муки и воды",
        "молоко получают от коровы",
        "сыр делают из молока",
        "масло делают из молока коровы",
        "хлеб и сыр хорошо сочетаются",
        "яблоко и груша фрукты сада",
        "яблоко растет на яблоне в саду",
        "груша сладкий фрукт как яблоко",
        "молоко и хлеб продукты завтрака",
        "сыр и масло молочные продукты",
    ],
    "tech": [
        "компьютер обрабатывает данные быстро",
        "программист пишет код на компьютере",
        "телефон и компьютер электронные устройства",
        "интернет соединяет компьютеры по всему миру",
        "программа работает на компьютере пользователя",
        "телефон использует интернет для связи",
        "код программы содержит ошибку",
        "программист исправил ошибку в коде",
        "компьютер и телефон нужны для работы",
        "интернет ускорил передачу данных",
    ],
}

sentences_raw = []
for topic, sents in TEMPLATES.items():
    sentences_raw.extend(sents)

# Дублируем корпус 6 раз с перемешиванием ПОРЯДКА предложений (не слов внутри
# предложения). Честная оговорка: это НЕ аугментация данных — новых наблюдений
# не появляется, все счётчики пар (слово, контекст) просто умножаются на 6.
# По сути это эквивалентно увеличению числа эпох обучения примерно в 6 раз
# (с другим порядком обхода примеров); делаем так только для удобства, чтобы
# внутри одной "эпохи" по объединённому списку модель видела больше шагов SGD.
augmented = []
for _ in range(6):
    shuffled = sentences_raw.copy()
    random.shuffle(shuffled)
    augmented.extend(shuffled)

corpus_tokens = [s.split() for s in augmented]
print("Число предложений в корпусе:", len(corpus_tokens))
print("Пример предложений:")
for s in corpus_tokens[:3]:
    print(" ", " ".join(s))


In [ ]:
# Построение словаря и таблицы частот для negative sampling
word_counts = Counter()
for sent in corpus_tokens:
    word_counts.update(sent)

MIN_COUNT = 1
vocab = [w for w, c in word_counts.items() if c >= MIN_COUNT]
vocab.sort()
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)

freqs = np.array([word_counts[w] for w in vocab], dtype=np.float64)
unigram_dist = freqs / freqs.sum()

# Сглаженное распределение для negative sampling: P_n(w) \propto U(w)^{3/4}
neg_dist = freqs ** 0.75
neg_dist = neg_dist / neg_dist.sum()

print("Размер словаря:", V)
print("Топ-10 частотных слов:", sorted(word_counts.items(), key=lambda x: -x[1])[:10])


## Демо-задача 1. SGNS «с нуля» на NumPy

Реализуем полный цикл обучения SGNS без готовых ML-библиотек: генерация пар (слово, контекст) в скользящем окне, выборка $k$ негативных примеров из сглаженного униграммного распределения, форвард и градиентный шаг по формулам из теории.

**Baseline для сравнения:** случайные (неинициализированные / не обученные) векторы той же размерности — проверяем, что после обучения косинусная близость похожих по смыслу слов (например, «король» и «королева») ощутимо выше, чем у случайных векторов.


In [ ]:
# Генерация обучающих пар (слово, контекст) в окне WINDOW
WINDOW = 3

def make_pairs(tokenized_sents, word2idx, window):
    pairs = []
    for sent in tokenized_sents:
        idxs = [word2idx[w] for w in sent if w in word2idx]
        n = len(idxs)
        for i, w in enumerate(idxs):
            lo, hi = max(0, i - window), min(n, i + window + 1)
            for j in range(lo, hi):
                if j != i:
                    pairs.append((w, idxs[j]))
    return pairs

train_pairs = make_pairs(corpus_tokens, word2idx, WINDOW)
print("Число обучающих пар (слово, контекст):", len(train_pairs))


In [ ]:
# Собственная реализация SGNS на NumPy
class SGNSNumpy:
    def __init__(self, vocab_size, dim, neg_dist, k=5, lr=0.05, seed=SEED):
        rng = np.random.RandomState(seed)
        # v_w -- "входные" (word) векторы, v_c -- "выходные" (context) векторы
        self.V_in = rng.normal(0, 0.1, size=(vocab_size, dim))
        self.V_out = rng.normal(0, 0.1, size=(vocab_size, dim))
        self.neg_dist = neg_dist
        self.k = k
        self.lr = lr
        self.rng = rng
        self.vocab_size = vocab_size

    @staticmethod
    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

    def sample_negatives(self, batch_size):
        return self.rng.choice(self.vocab_size, size=(batch_size, self.k), p=self.neg_dist)

    def train_step(self, w_idx, c_idx):
        neg_idx = self.sample_negatives(len(w_idx))  # (B, k)
        v_w = self.V_in[w_idx]                       # (B, d)
        v_c = self.V_out[c_idx]                       # (B, d)
        v_neg = self.V_out[neg_idx]                    # (B, k, d)

        pos_score = np.sum(v_w * v_c, axis=1)          # (B,)
        pos_sig = self.sigmoid(pos_score)
        loss_pos = -np.log(pos_sig + 1e-10)

        neg_score = np.einsum("bd,bkd->bk", v_w, v_neg)  # (B, k)
        neg_sig = self.sigmoid(neg_score)
        loss_neg = -np.sum(np.log(1 - neg_sig + 1e-10), axis=1)

        loss = np.mean(loss_pos + loss_neg)

        # Градиенты по формулам из теории
        grad_vw = (pos_sig - 1)[:, None] * v_c + np.einsum("bk,bkd->bd", neg_sig, v_neg)
        grad_vc_pos = (pos_sig - 1)[:, None] * v_w
        grad_vneg = neg_sig[:, :, None] * v_w[:, None, :]

        # SGD-обновление (накопление через np.add.at из-за возможных повторов индексов)
        np.add.at(self.V_in, w_idx, -self.lr * grad_vw)
        np.add.at(self.V_out, c_idx, -self.lr * grad_vc_pos)
        for b in range(len(w_idx)):
            np.add.at(self.V_out, neg_idx[b], -self.lr * grad_vneg[b])

        return loss

    def get_embedding(self, idx):
        # Итоговый вектор слова — усреднение входного и выходного представлений
        return (self.V_in[idx] + self.V_out[idx]) / 2.0


DIM = 30
K_NEG = 5
N_EPOCHS = 40
BATCH = 64

model_np = SGNSNumpy(V, DIM, neg_dist, k=K_NEG, lr=0.05, seed=SEED)

pairs_arr = np.array(train_pairs)
losses_np = []
rng_shuffle = np.random.RandomState(SEED)
for epoch in range(N_EPOCHS):
    perm = rng_shuffle.permutation(len(pairs_arr))
    epoch_losses = []
    for start in range(0, len(perm), BATCH):
        batch_idx = perm[start:start + BATCH]
        w_idx = pairs_arr[batch_idx, 0]
        c_idx = pairs_arr[batch_idx, 1]
        loss = model_np.train_step(w_idx, c_idx)
        epoch_losses.append(loss)
    losses_np.append(np.mean(epoch_losses))

print("Loss в начале обучения:", round(losses_np[0], 4))
print("Loss в конце обучения:", round(losses_np[-1], 4))


In [ ]:
# Baseline: случайные (необученные) векторы -- сравнение косинусной близости
def cos_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

rng_base = np.random.RandomState(SEED)
random_embeddings = rng_base.normal(0, 0.1, size=(V, DIM))

pairs_to_check = [("король", "королева"), ("кошка", "собака"), ("москва", "париж"),
                   ("хлеб", "молоко"), ("компьютер", "программист")]

rows = []
for a, b in pairs_to_check:
    if a in word2idx and b in word2idx:
        ia, ib = word2idx[a], word2idx[b]
        sim_random = cos_sim(random_embeddings[ia], random_embeddings[ib])
        sim_sgns = cos_sim(model_np.get_embedding(ia), model_np.get_embedding(ib))
        rows.append({"пара": f"{a}-{b}", "cos (случайные, baseline)": round(sim_random, 3),
                     "cos (SGNS NumPy)": round(sim_sgns, 3)})

df_demo1 = pd.DataFrame(rows)
print(df_demo1.to_string(index=False))
print()
print("Средняя близость (baseline, случайные):", round(df_demo1["cos (случайные, baseline)"].mean(), 3))
print("Средняя близость (SGNS NumPy):", round(df_demo1["cos (SGNS NumPy)"].mean(), 3))


**Вывод.** После обучения SGNS средняя косинусная близость семантически связанных пар слов (внутри одной темы: «король-королева», «кошка-собака», «москва-париж» и т.д.) заметно выше, чем у случайных необученных векторов той же размерности, — это подтверждает, что градиентные шаги SGNS действительно «сближают» векторы слов, встречающихся в похожих контекстах, в соответствии с гипотезой дистрибутивности.


## Демо-задача 2. Skip-gram на PyTorch (аналог библиотечного обучения) и сравнение с NumPy-реализацией

Поскольку `gensim` недоступен в среде (см. замечание в шапке), library-grade обучение воспроизведено на PyTorch: `nn.Embedding` для входных/выходных векторов, `BCEWithLogitsLoss` для той же цели SGNS, векторизованное обучение на GPU/CPU. Сравним top-3 ближайших соседей для нескольких опорных слов между NumPy- и PyTorch-реализациями — они должны быть качественно похожими (не идентичными из-за случайности инициализации и разного порядка обновлений).


In [ ]:
if TORCH_OK:
    class SGNSTorch(nn.Module):
        def __init__(self, vocab_size, dim):
            super().__init__()
            self.in_emb = nn.Embedding(vocab_size, dim)
            self.out_emb = nn.Embedding(vocab_size, dim)
            nn.init.normal_(self.in_emb.weight, 0, 0.1)
            nn.init.normal_(self.out_emb.weight, 0, 0.1)

        def forward(self, w_idx, c_idx, neg_idx):
            v_w = self.in_emb(w_idx)             # (B, d)
            v_c = self.out_emb(c_idx)            # (B, d)
            v_neg = self.out_emb(neg_idx)        # (B, k, d)
            pos_score = (v_w * v_c).sum(dim=1)                 # (B,)
            neg_score = torch.einsum("bd,bkd->bk", v_w, v_neg)  # (B, k)
            return pos_score, neg_score

        def embedding(self, idx):
            with torch.no_grad():
                return ((self.in_emb.weight[idx] + self.out_emb.weight[idx]) / 2.0).numpy()

    torch.manual_seed(SEED)
    model_pt = SGNSTorch(V, DIM)
    optimizer = torch.optim.Adam(model_pt.parameters(), lr=0.01)
    # ВАЖНО: суммируем по элементам и делим на размер батча вручную.
    # BCEWithLogitsLoss(reduction='mean') на neg_score формы (B, k) усреднял бы
    # по B*k элементам, то есть взвешивал бы каждый негатив с весом 1/k --
    # это НЕ соответствует теоретической функции потерь SGNS, где k негативов
    # входят в сумму с единичными весами наравне с позитивной парой.
    bce_sum = nn.BCEWithLogitsLoss(reduction="sum")

    neg_dist_t = torch.tensor(neg_dist, dtype=torch.float64)
    pairs_t = torch.tensor(train_pairs, dtype=torch.long)

    losses_pt = []
    gen = torch.Generator().manual_seed(SEED)
    for epoch in range(N_EPOCHS):
        perm = torch.randperm(len(pairs_t), generator=gen)
        epoch_losses = []
        for start in range(0, len(perm), BATCH):
            batch_idx = perm[start:start + BATCH]
            w_idx = pairs_t[batch_idx, 0]
            c_idx = pairs_t[batch_idx, 1]
            neg_idx = torch.multinomial(neg_dist_t, len(w_idx) * K_NEG, replacement=True).view(len(w_idx), K_NEG)

            pos_score, neg_score = model_pt(w_idx, c_idx, neg_idx)
            pos_target = torch.ones_like(pos_score)
            neg_target = torch.zeros_like(neg_score)

            # Сумма по позитиву и всем k негативам (веса как в теории), затем среднее по батчу
            loss = (bce_sum(pos_score, pos_target) + bce_sum(neg_score, neg_target)) / len(w_idx)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        losses_pt.append(np.mean(epoch_losses))

    print("PyTorch SGNS loss в начале:", round(losses_pt[0], 4))
    print("PyTorch SGNS loss в конце:", round(losses_pt[-1], 4))
else:
    print("PyTorch недоступен -- пропускаем демо-задачу 2 (используем только NumPy-реализацию).")


In [ ]:
def nearest_neighbors(embeddings, word, word2idx, idx2word, topn=3):
    if word not in word2idx:
        return []
    idx = word2idx[word]
    v = embeddings[idx]
    sims = []
    for j in range(len(embeddings)):
        if j == idx:
            continue
        sims.append((idx2word[j], cos_sim(v, embeddings[j])))
    sims.sort(key=lambda x: -x[1])
    return sims[:topn]

emb_np_all = np.array([model_np.get_embedding(i) for i in range(V)])
if TORCH_OK:
    emb_pt_all = model_pt.embedding(torch.arange(V))
else:
    emb_pt_all = emb_np_all  # fallback, чтобы код ниже не падал

probe_words = ["король", "кошка", "москва", "хлеб", "компьютер"]
rows = []
for w in probe_words:
    nn_np = nearest_neighbors(emb_np_all, w, word2idx, idx2word, topn=3)
    nn_pt = nearest_neighbors(emb_pt_all, w, word2idx, idx2word, topn=3)
    rows.append({
        "слово": w,
        "соседи (NumPy SGNS)": ", ".join(f"{s}({c:.2f})" for s, c in nn_np),
        "соседи (PyTorch SGNS)" if TORCH_OK else "соседи (PyTorch, fallback=NumPy)":
            ", ".join(f"{s}({c:.2f})" for s, c in nn_pt),
    })

df_demo2 = pd.DataFrame(rows)
print(df_demo2.to_string(index=False))


**Вывод.** Обе реализации (NumPy «с нуля» и PyTorch — аналог библиотечного обучения при отсутствии `gensim`) находят для каждого опорного слова соседей преимущественно из той же тематической группы корпуса (например, для «король» — «королева», «принц», «царь»), что подтверждает согласованность самописной реализации с обучением через автоматическое дифференцирование PyTorch: главное — правильная целевая функция и семплирование негативов, а не конкретный фреймворк.


## Демо-задача 3. Baseline матричной факторизации: PPMI + SVD

По теореме Levy & Goldberg (см. теорию) SGNS неявно факторизует SPPMI-матрицу. Проверим это напрямую: построим PPMI-матрицу совместных встреч слово-слово в окне, обрежем отрицательные значения (получим PPMI = $\max(0, \mathrm{PMI})$) и найдём её низкоранговое приближение через SVD. Это полностью детерминированный (без случайных градиентных шагов) baseline того же семантического пространства.


In [ ]:
# Матрица совместных встреч (co-occurrence) в окне WINDOW
cooc = np.zeros((V, V), dtype=np.float64)
for sent in corpus_tokens:
    idxs = [word2idx[w] for w in sent if w in word2idx]
    n = len(idxs)
    for i, w in enumerate(idxs):
        lo, hi = max(0, i - WINDOW), min(n, i + WINDOW + 1)
        for j in range(lo, hi):
            if j != i:
                cooc[w, idxs[j]] += 1.0

total = cooc.sum()
word_totals = cooc.sum(axis=1)

# PMI(w,c) = log( #(w,c) * |D| / (#(w) * #(c)) ), PPMI = max(0, PMI)
with np.errstate(divide="ignore", invalid="ignore"):
    pmi = np.log((cooc * total + 1e-10) / (np.outer(word_totals, word_totals) + 1e-10))
ppmi = np.maximum(pmi, 0.0)
ppmi[np.isnan(ppmi)] = 0.0

print("Форма PPMI-матрицы:", ppmi.shape)
print("Доля ненулевых элементов PPMI:", round((ppmi > 0).mean(), 3))


In [ ]:
# SVD-факторизация PPMI-матрицы, d = DIM компонент
from numpy.linalg import svd

U, S, Vt = svd(ppmi, full_matrices=False)
emb_svd_all = U[:, :DIM] * np.sqrt(S[:DIM])[None, :]

rows = []
for w in probe_words:
    nn_svd = nearest_neighbors(emb_svd_all, w, word2idx, idx2word, topn=3)
    rows.append({"слово": w, "соседи (PPMI-SVD baseline)": ", ".join(f"{s}({c:.2f})" for s, c in nn_svd)})

df_demo3 = pd.DataFrame(rows)
print(df_demo3.to_string(index=False))


In [ ]:
# Итоговая сравнительная таблица трёх методов для одних и тех же опорных слов
compare_rows = []
for w in probe_words:
    compare_rows.append({
        "слово": w,
        "SGNS NumPy (топ-1)": nearest_neighbors(emb_np_all, w, word2idx, idx2word, topn=1)[0][0],
        "SGNS PyTorch (топ-1)": nearest_neighbors(emb_pt_all, w, word2idx, idx2word, topn=1)[0][0],
        "PPMI-SVD (топ-1)": nearest_neighbors(emb_svd_all, w, word2idx, idx2word, topn=1)[0][0],
        "случайные, baseline (топ-1)": nearest_neighbors(random_embeddings, w, word2idx, idx2word, topn=1)[0][0],
    })
df_compare = pd.DataFrame(compare_rows)
print(df_compare.to_string(index=False))


**Вывод.** PPMI-SVD (детерминированный матричный метод) и SGNS (стохастическое обучение) находят похожих ближайших соседей для большинства опорных слов, что на практике подтверждает теорему Levy & Goldberg о неявной факторизации SPPMI-матрицы через SGNS. Baseline на случайных векторах, ожидаемо, не даёт содержательных соседей — топ-1 сосед для случайных векторов не связан по смыслу с опорным словом.


## Демо-задача 4. Лингвистические аналогии (3CosAdd)

Проверим аналогии вида «$a$ относится к $b$, как $c$ относится к $d$» по формуле 3CosAdd:
$$
\hat d = \arg\max_{d' \in V\setminus\{a,b,c\}} \cos\left(v_b - v_a + v_c,\; v_{d'}\right).
$$
Из-за малого размера учебного корпуса ожидается ограниченная точность (обучающего сигнала мало) — это ожидаемо и подчёркивается в выводе; на реальных корпусах в млн токенов (например, Lenta.ru, см. задачи для самостоятельного решения) точность заметно выше.


In [ ]:
def analogy_3cosadd(a, b, c, embeddings, word2idx, idx2word, exclude=None, topn=5):
    if exclude is None:
        exclude = {a, b, c}
    if not all(w in word2idx for w in (a, b, c)):
        return []
    va, vb, vc = embeddings[word2idx[a]], embeddings[word2idx[b]], embeddings[word2idx[c]]
    target = vb - va + vc
    sims = []
    for j in range(len(embeddings)):
        w = idx2word[j]
        if w in exclude:
            continue
        sims.append((w, cos_sim(target, embeddings[j])))
    sims.sort(key=lambda x: -x[1])
    return sims[:topn]

analogy_tests = [
    ("король", "королева", "принц", "принцесса"),   # мужчина:женщина по роли
    ("москва", "россия", "париж", "франция"),        # столица:страна
    ("король", "принц", "царь", "царица"),            # аналог трона (нестрогая)
]

rows = []
correct = 0
for a, b, c, expected in analogy_tests:
    result = analogy_3cosadd(a, b, c, emb_np_all, word2idx, idx2word, topn=3)
    top1 = result[0][0] if result else None
    is_correct = int(top1 == expected)
    correct += is_correct
    rows.append({
        "a:b :: c:?": f"{a}:{b} :: {c}:?",
        "ожидание": expected,
        "топ-3 предсказания": ", ".join(f"{w}({s:.2f})" for w, s in result),
        "top-1 верно": bool(is_correct),
    })

df_analogy = pd.DataFrame(rows)
print(df_analogy.to_string(index=False))
print()
top1_accuracy = correct / len(analogy_tests)
print(f"Top-1 accuracy по аналогиям (SGNS NumPy, наш маленький корпус): {top1_accuracy:.2f}")


**Вывод.** На этом маленьком синтетическом корпусе ожидается очень низкая top-1 accuracy (вплоть до $0$ — сравните с выводом ячейки выше): метод 3CosAdd на таком корпусе систематически предпочитает грамматически связанные словоформы (например, «россии», «францией») вместо целевого слова в нужной форме. Причина — (а) отсутствие лемматизации: русский язык флективен, и «россия»/«россии»/«россией» — разные токены с разными векторами; (б) крайне малый объём обучающих данных (сотни предложений вместо миллионов токенов). Это ожидаемый и содержательный негативный результат: он иллюстрирует, почему в задаче 3 из раздела самостоятельных задач предлагается либо расширить корпус, либо явно учитывать словоформы при проверке аналогий. На корпусах в миллионы токенов (например, подкорпус Lenta.ru) и/или после лемматизации accuracy на стандартных аналогиях значительно выше (типично десятки процентов).


## Демо-задача 5. Визуализация эмбеддингов через t-SNE и UMAP

Спроецируем эмбеддинги всех слов словаря на плоскость двумя методами нелинейного снижения размерности и раскрасим точки по тематической группе (известной из построения корпуса) — это позволяет визуально оценить, насколько хорошо обученные эмбеддинги группируют семантически близкие слова.


In [ ]:
from sklearn.manifold import TSNE

# Тематическая метка для каждого слова: тема, в шаблонах которой слово встречалось чаще
word_topic = {}
for topic, sents in TEMPLATES.items():
    for s in sents:
        for w in s.split():
            word_topic.setdefault(w, Counter())[topic] += 1
word_to_main_topic = {w: cnt.most_common(1)[0][0] for w, cnt in word_topic.items()}

labels = [word_to_main_topic.get(w, "other") for w in vocab]

tsne = TSNE(n_components=2, random_state=SEED, perplexity=min(15, max(2, V // 4)),
            init="pca", learning_rate="auto")
coords_tsne = tsne.fit_transform(emb_np_all)
print("t-SNE проекция построена, форма:", coords_tsne.shape)


In [ ]:
try:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=min(10, V - 1))
    coords_umap = reducer.fit_transform(emb_np_all)
    UMAP_OK = True
    print("UMAP проекция построена, форма:", coords_umap.shape)
except ImportError:
    UMAP_OK = False
    coords_umap = None
    print("Пакет umap-learn не установлен: pip install umap-learn. Используем только t-SNE.")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2 if UMAP_OK else 1, figsize=(14, 6))
if not UMAP_OK:
    axes = [axes]

topics_unique = sorted(set(labels))
cmap = plt.get_cmap("tab10")
color_map = {t: cmap(i / max(1, len(topics_unique))) for i, t in enumerate(topics_unique)}

ax = axes[0]
for t in topics_unique:
    idxs = [i for i, lb in enumerate(labels) if lb == t]
    ax.scatter(coords_tsne[idxs, 0], coords_tsne[idxs, 1], label=t, s=40, color=color_map[t])
for i, w in enumerate(vocab):
    ax.annotate(w, (coords_tsne[i, 0], coords_tsne[i, 1]), fontsize=6, alpha=0.7)
ax.set_title("t-SNE проекция эмбеддингов (SGNS NumPy)")
ax.legend(fontsize=8)

if UMAP_OK:
    ax = axes[1]
    for t in topics_unique:
        idxs = [i for i, lb in enumerate(labels) if lb == t]
        ax.scatter(coords_umap[idxs, 0], coords_umap[idxs, 1], label=t, s=40, color=color_map[t])
    for i, w in enumerate(vocab):
        ax.annotate(w, (coords_umap[i, 0], coords_umap[i, 1]), fontsize=6, alpha=0.7)
    ax.set_title("UMAP проекция эмбеддингов (SGNS NumPy)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("seminar07_tsne_umap.png", dpi=110)
plt.show()
print("Рисунок сохранён: seminar07_tsne_umap.png")


**Вывод.** На карте t-SNE/UMAP слова из одной тематической группы корпуса (например, «королевская семья», «животные», «столицы», «еда», «технологии») образуют визуально различимые кластеры, что подтверждает: обученные SGNS-эмбеддинги отражают тематическую/семантическую структуру корпуса, а не случайное расположение. Расстояния между кластерами на графике не следует интерпретировать буквально (t-SNE/UMAP сохраняют только локальную структуру).


## Задачи для самостоятельного решения

Выбраны 5 из 7 задач силлабуса семинара 7. Для каждой задачи: реализуйте код в ячейке с `# TODO`, посчитайте указанную метрику, сравните с baseline (где применимо) и сформулируйте вывод в отдельной markdown-ячейке (допишите текст под «Вывод:»).


### Задача 1 (теоретическая). Градиент функции потерь SGNS

Выведите градиент функции потерь SGNS
$$
\mathcal{L}(w,c) = \log\sigma(v_w^\top v_c) + \sum_{i=1}^{k}\log\sigma(-v_w^\top v_{c_i})
$$
по вектору $v_w$ для одной положительной пары $(w,c)$ и $k$ негативных примеров $c_1,\dots,c_k$. Запишите промежуточный шаг (производную $\sigma$) и итоговую формулу.

**Ожидаемый диапазон самопроверки:** итоговая формула должна совпадать с
$\frac{\partial \mathcal{L}}{\partial v_w} = (1-\sigma(v_w^\top v_c))v_c - \sum_i \sigma(v_w^\top v_{c_i})v_{c_i}$
(с точностью до знака при формулировке как minimization/maximization) — сверьте с кодом `SGNSNumpy.train_step` выше.


In [ ]:
# TODO: Задача 1 -- выведите градиент аналитически (можно в этой ячейке дать вывод как
# комментарий или проверить численно через конечные разности).
#
# Подсказка: d/dx log(sigma(x)) = 1 - sigma(x); d/dx log(sigma(-x)) = -sigma(x).
#
# Численная проверка (сравнение аналитического градиента с конечными разностями):
def numeric_grad_check(v_w, v_c, v_negs, eps=1e-5):
    # TODO: реализуйте численную проверку градиента по v_w с помощью центральных разностей
    # и сравните со значением, которое даёт формула из train_step.
    raise NotImplementedError

# TODO: вызовите numeric_grad_check на случайных векторах и выведите максимальную
# абсолютную разницу между численным и аналитическим градиентом (ожидается < 1e-4).


### Задача 2 (теоретическая). Сглаживание частот в negative sampling

Покажите (аналитически и на числовом примере на этом корпусе), что распределение негативной выборки $P_n(w)\propto U(w)^{3/4}$ сглаживает частоты по сравнению с униграммным распределением $U(w)$, и объясните, зачем это нужно.

**Ожидаемый диапазон самопроверки:** после возведения в степень $3/4$ и перенормировки отношение вероятностей самого частого и самого редкого слова должно **уменьшиться** по сравнению с исходным униграммным распределением (обычно в несколько раз, конкретная величина зависит от распределения частот корпуса).


In [ ]:
# TODO: Задача 2 -- сравните на числах unigram_dist и neg_dist, уже посчитанные выше.
# 1. Найдите отношение P(самое частое слово) / P(самое редкое слово) для unigram_dist.
# 2. Найдите то же отношение для neg_dist.
# 3. Убедитесь, что второе отношение меньше первого (сглаживание), выведите оба числа.
most_frequent_idx = None  # TODO
least_frequent_idx = None  # TODO
ratio_unigram = None  # TODO
ratio_neg = None  # TODO
print("Отношение частот (unigram):", ratio_unigram)
print("Отношение частот (P_n^0.75):", ratio_neg)


### Задача 3. Набор русскоязычных аналогий и accuracy top-1/top-5

Постройте набор из **не менее 8 аналогий** на русском языке (страны-столицы, мужской-женский род — используйте слова из корпуса семинара или расширьте корпус собственными предложениями по тем же тематическим шаблонам) и посчитайте top-1 и top-5 accuracy по формуле 3CosAdd, используя обученные эмбеддинги `emb_np_all`.

**Ожидаемый диапазон самопроверки:** как показала демо-задача 4 выше, на самом узком исходном корпусе top-1 accuracy типично близка к **0.0–0.2** из-за флективности русского языка (разные словоформы — разные векторы); top-5 accuracy ожидается выше (обычно **0.2–0.5**), так как среди топ-5 часто попадают родственные словоформы целевого слова. Формально верным считается только точное совпадение токена (без лемматизации); расширив корпус своими шаблонами или добавив лемматизацию (например, вручную для известных падежных форм), можно добиться более высоких значений — это само по себе содержательный вывод. Если ваш top-5 accuracy тоже равен 0 — убедитесь, что все слова аналогии есть в словаре `vocab` и что целевое слово (в той же словоформе) встречалось в корпусе.


In [ ]:
# TODO: Задача 3 -- составьте список analogy_set = [(a, b, c, expected_d), ...] (>= 8 троек)
analogy_set = [
    # ("москва", "россия", "берлин", "германия"),
    # TODO: добавьте не менее 8 аналогий
]

# TODO: для каждой аналогии вызовите analogy_3cosadd(..., topn=5) и посчитайте:
#   top1_hits -- число случаев, когда topn=5 результат[0][0] == expected_d
#   top5_hits -- число случаев, когда expected_d встречается среди топ-5
top1_accuracy = None  # TODO
top5_accuracy = None  # TODO
print("Top-1 accuracy:", top1_accuracy)
print("Top-5 accuracy:", top5_accuracy)

# TODO: сравните top1_accuracy со случайным baseline = 1 / (V - 3) (случайный выбор слова)
random_baseline = None  # TODO
print("Случайный baseline (top-1):", random_baseline)


### Задача 4. Визуализация 300 наиболее частых слов с раскраской по частям речи

Возьмите (или расширьте синтетический корпус до) не менее 300 уникальных слов, обучите SGNS, визуализируйте через t-SNE, раскрасив точки по **грубой части речи** (существительное/глагол/прочее — можно определить эвристически по окончаниям или заранее заданным спискам). Опишите наблюдаемую структуру: группируются ли слова по частям речи, по темам, или структура смешанная.

**Ожидаемый диапазон самопроверки:** если данных недостаточно (корпус сильно меньше 300 уникальных слов), допустимо использовать весь словарь `vocab` данного семинара и явно указать это ограничение в выводе; итоговый график должен содержать подписи слов и легенду по цвету.


In [ ]:
# TODO: Задача 4 -- при необходимости расширьте corpus_tokens / vocab, переобучите SGNS
# (можно переиспользовать SGNSNumpy с новыми данными), затем:
# 1. Постройте эвристическую разметку части речи для слов словаря (например, по окончаниям
#    -ать/-ить/-еть -> глагол, иначе -> существительное/прочее).
# 2. Постройте t-SNE проекцию, раскрасьте по части речи, сохраните рисунок в файл.
pos_labels = {}  # TODO: word -> "noun"/"verb"/"other"

# TODO: постройте график и сохраните как seminar07_task4_pos.png


### Задача 5. Косинусная близость синонимов vs случайных пар слов

Сравните косинусную близость пар слов из одной тематической группы (аналог «синонимов» в этом учебном корпусе — например, слова, регулярно встречающиеся в одинаковом контексте) со случайными парами слов из разных тематических групп; постройте распределения (гистограммы) обеих величин и сопоставьте их средние значения.

**Ожидаемый диапазон самопроверки:** средняя косинусная близость «синонимичных»/тематически близких пар должна быть выше средней близости случайных межтематических пар минимум на **0.05–0.15** по абсолютной величине на этом корпусе (конкретная величина зависит от качества обучения); если разница отрицательна или нулевая — проверьте число эпох обучения и размер корпуса.


In [ ]:
# TODO: Задача 5
# 1. Сформируйте список пар слов из одной темы (same_topic_pairs) и пар из разных тем
#    (random_pairs), например, используя word_to_main_topic.
# 2. Посчитайте cos_sim для каждой пары в обеих группах.
# 3. Постройте гистограммы распределений (plt.hist) и выведите средние значения.
same_topic_pairs = []   # TODO
random_pairs = []       # TODO

same_topic_sims = []    # TODO
random_sims = []        # TODO

print("Средняя близость (одна тема):", None)   # TODO
print("Средняя близость (случайные межтемные пары):", None)  # TODO


### Задача 6* (опциональная). Реализуйте CBOW

Опираясь на теоретическое дополнение о CBOW выше и на класс `SGNSNumpy` (или `SGNSTorch`), реализуйте обучение CBOW с негативной выборкой на этом же корпусе: для каждой позиции усредните векторы контекстных слов окна и предсказывайте центральное слово. Сравните ближайших соседей опорных слов (`probe_words`) у CBOW и skip-gram.

**Ожидаемый диапазон самопроверки:** качественно соседи должны совпадать с SGNS по тематическим группам; на таком маленьком корпусе skip-gram обычно даёт чуть более «чистых» соседей для редких слов — если различий не видно, это тоже допустимый результат, зафиксируйте его в выводе.


In [ ]:
# TODO (опционально): Задача 6* -- реализация CBOW с негативной выборкой
# 1. Постройте обучающие примеры (контекстное окно C, центральное слово w) по corpus_tokens.
# 2. Модифицируйте SGNSNumpy/SGNSTorch: вместо вектора v_w используйте среднее векторов слов окна.
# 3. Обучите модель и сравните nearest_neighbors для probe_words с результатами skip-gram.

pass  # замените pass своим решением (задача опциональная)


## Чек-лист оценивания

- [ ] Данные и обучающие пары построены корректно (окно контекста, negative sampling из сглаженного распределения $U(w)^{3/4}$).
- [ ] Присутствует явный **baseline** (случайные векторы) для каждой демо-задачи, где это применимо.
- [ ] Метрики качества посчитаны по формулам из теории (косинусная близость, top-1/top-5 accuracy аналогий) — не «на глаз».
- [ ] Результаты сведены в сравнительную таблицу (SGNS NumPy / PyTorch / PPMI-SVD / baseline).
- [ ] Зафиксирован `SEED`, обучение воспроизводимо при повторном запуске.
- [ ] Для каждой демо-задачи и задачи самостоятельного решения есть явный текстовый вывод (интерпретация числа/графика, а не просто печать результата).
- [ ] Для задач самостоятельного решения указан ожидаемый диапазон качества и авторский результат сопоставлен с ним.
